# **Document Question Answering System using Retrieval-Augmented Generation (RAG)**

## Overview

This project implements a **Retrieval-Augmented Generation (RAG)** system that answers questions from custom PDF documents. It retrieves relevant information using semantic search and generates accurate, context-aware answers with **Google Gemini**.

---

## Objectives

- Build a simple RAG-based document question answering system.
- Process and index PDF documents.
- Generate semantic embeddings.
- Retrieve relevant information using FAISS.
- Generate accurate answers using Google Gemini.

---

## Key Concepts

- **Retrieval:** Finds the most relevant document chunks using semantic similarity search.
- **Augmentation:** Combines the retrieved context with the user's query.
- **Generation:** Uses Google Gemini to generate document-based answers.

---

## System Architecture

1. Load PDF document.
2. Split text into chunks.
3. Generate embeddings.
4. Store embeddings in FAISS.
5. Process user query.
6. Retrieve relevant chunks.
7. Generate the final answer.

---

## Dataset

The system works with custom documents such as:

- PDF Documents
- Notes
- Research Papers
- Books
- Articles

---

## Components Used

- **PyPDF** – PDF text extraction
- **Sentence Transformers** – Text embeddings
- **FAISS** – Vector similarity search
- **Google Gemini** – Answer generation

---

## Workflow

1. Load and preprocess the document.
2. Split the text into chunks.
3. Generate embeddings.
4. Store embeddings in FAISS.
5. Retrieve relevant document chunks.
6. Generate the final answer using Google Gemini.

# **Step 1: Install and Import Required Libraries**

In this step, we install (if required) and import all the libraries needed to build the Retrieval-Augmented Generation (RAG) system.

### Libraries Used

- **pypdf** – Extracts text from PDF documents.
- **sentence-transformers** – Generates semantic embeddings for document chunks.
- **faiss-cpu** – Performs fast vector similarity search.
- **google-generativeai** – Connects to the Google Gemini Large Language Model.
- **langchain-text-splitters** – Splits large documents into smaller overlapping chunks.
- **NumPy** – Supports numerical operations and embedding manipulation.
- **os, re, getpass** – Used for file handling, text preprocessing, and secure API key input.

After importing all the required libraries, the notebook is ready for document processing and question answering.

In [3]:
import os
import re
import getpass
import numpy as np
import pypdf
import faiss

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

import google.generativeai as genai

print("All libraries imported successfully!")

All libraries imported successfully!


# **Step 2: Configure Google Gemini API**

In this step, the Google Gemini API is configured so that the Large Language Model (LLM) can generate answers based on the retrieved document context.

The API key is obtained securely using one of the following methods:

- **Environment Variable:** If the `GEMINI_API_KEY` environment variable is available, it is used automatically.
- **Manual Input:** If the environment variable is not found, the user is prompted to enter the API key securely using `getpass()`, which hides the input while typing.

Once the API key is validated, the Google Generative AI client is initialized and is ready to process user queries.

> **Note:** Clear the notebook output before sharing or submitting it to avoid exposing your API key.

In [ ]:
# ============================================================
# Step 2: Configure Google Gemini API
# ============================================================

# Security Note:
# Clear the output of this cell before submitting the notebook
# to avoid exposing your API key.

def configure_gemini_api(api_key=None):
    """
    Configures the Google Gemini API.

    The API key is loaded from the environment variable
    'GEMINI_API_KEY'. If it is not found, the user is prompted
    to enter the key securely.

    Args:
        api_key (str, optional): Google Gemini API Key.

    Returns:
        str: Configured API key.
    """

    # Check whether an API key was provided
    if not api_key:

        # Try reading the API key from environment variables
        api_key = os.environ.get("GEMINI_API_KEY")

        # If the environment variable is not available,
        # ask the user to enter the key securely.
        if not api_key:
            print("GEMINI_API_KEY environment variable not found.")

            api_key = getpass.getpass(
                "Enter your Google Gemini API Key: "
            )

    # Validate the API key
    if not api_key.strip():
        raise ValueError("API Key cannot be empty.")

    # Configure the Google Generative AI client
    genai.configure(api_key=api_key)

    print("Gemini API configured successfully!")

    return api_key


# ============================================================
# Initialize Gemini API
# ============================================================

try:
    gemini_api_key = configure_gemini_api()

except Exception as error:
    print(f"Configuration failed: {error}")

# **Step 3: Load and Preprocess the PDF Document**

Before a Retrieval-Augmented Generation (RAG) system can answer questions, the uploaded PDF document must be processed into a format suitable for semantic search.

This preprocessing stage consists of three main tasks:

### 1. Load the PDF
- Open the PDF document using the **PyPDF** library.
- Extract text from each page individually.
- Verify that the document contains readable text.

### 2. Clean the Extracted Text
- Remove unnecessary spaces and tabs.
- Normalize multiple blank lines.
- Improve text quality for embedding generation.

### 3. Split the Text into Chunks
- Divide the document into smaller overlapping text chunks.
- Preserve the page number of each chunk for source attribution.
- Create manageable pieces of text for semantic retrieval.

These processed chunks are later converted into embeddings and stored in a FAISS vector database for efficient similarity search.

In [5]:
# ============================================================
# Step 3: Load and Preprocess the PDF Document
# ============================================================

# ------------------------------------------------------------
# Function: Load PDF
# Reads the PDF file and extracts text from each page.
# ------------------------------------------------------------

def load_pdf(pdf_path: str) -> list[str]:
    """
    Loads a PDF document and extracts text page by page.

    Args:
        pdf_path (str): Path to the PDF document.

    Returns:
        list[str]: A list containing the extracted text from each page.
    """

    # Check whether the PDF file exists
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF document not found: {pdf_path}")

    # Read the PDF document
    reader = pypdf.PdfReader(pdf_path)

    pages_text = []

    # Extract text from every page
    for page in reader.pages:

        text = page.extract_text()

        # Store extracted text (empty string if no text found)
        pages_text.append(text if text else "")

    # Verify that the PDF contains readable text
    if not pages_text or all(not page.strip() for page in pages_text):
        raise ValueError("The PDF contains no extractable text.")

    print(f"Successfully loaded PDF with {len(pages_text)} pages.")

    return pages_text


# ------------------------------------------------------------
# Function: Clean Extracted Text
# Removes unnecessary spaces and formatting issues.
# ------------------------------------------------------------

def clean_text(pages_text: list[str]) -> list[str]:
    """
    Cleans the extracted text from the PDF.

    Args:
        pages_text (list[str]): Raw text extracted from each page.

    Returns:
        list[str]: Cleaned text.
    """

    cleaned_pages = []

    for text in pages_text:

        # Replace multiple spaces or tabs with a single space
        text = re.sub(r"[ \t]+", " ", text)

        # Replace multiple blank lines with two new lines
        text = re.sub(r"\n{3,}", "\n\n", text)

        # Remove leading and trailing whitespace
        cleaned_pages.append(text.strip())

    print("Text cleaning completed.")

    return cleaned_pages


# ------------------------------------------------------------
# Function: Split Text into Chunks
# Divides the document into smaller overlapping chunks.
# ------------------------------------------------------------

def split_text(
    cleaned_pages: list[str],
    chunk_size: int = 500,
    chunk_overlap: int = 50
) -> list[dict]:
    """
    Splits cleaned text into overlapping chunks.

    Args:
        cleaned_pages (list[str]): Cleaned document text.
        chunk_size (int): Maximum size of each text chunk.
        chunk_overlap (int): Number of overlapping characters.

    Returns:
        list[dict]: List of text chunks with page numbers.
    """

    # Create a text splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = []

    # Process each page individually
    for page_number, page_text in enumerate(cleaned_pages, start=1):

        # Skip empty pages
        if not page_text.strip():
            continue

        # Split the page into smaller chunks
        page_chunks = splitter.split_text(page_text)

        # Store each chunk along with its page number
        for chunk in page_chunks:
            chunks.append({
                "text": chunk,
                "page": page_number
            })

    print(f"Created {len(chunks)} text chunks.")

    return chunks

# **Step 4: Generate Embeddings and Build the FAISS Index**

After the document has been split into smaller text chunks, each chunk is converted into a numerical vector called an **embedding**. These embeddings capture the semantic meaning of the text, allowing the system to retrieve relevant information even when the exact keywords are not present.

This step consists of two main tasks:

### 1. Generate Embeddings
- Load the **Sentence Transformer** model.
- Convert each text chunk into a dense vector representation.
- Produce embeddings suitable for semantic similarity search.

### 2. Build the FAISS Index
- Convert embeddings to the required `float32` format.
- Normalize embeddings for cosine similarity.
- Store the embeddings in a **FAISS (Facebook AI Similarity Search)** index.
- Enable efficient retrieval of the most relevant document chunks based on a user's query.

The generated FAISS index serves as the knowledge base for the Retrieval-Augmented Generation (RAG) system.

In [6]:
# ============================================================
# Step 4: Generate Embeddings and Build the FAISS Index
# ============================================================

# ------------------------------------------------------------
# Function: Create Embeddings
# Converts text chunks into semantic vector representations
# using a Sentence Transformer model.
# ------------------------------------------------------------

def create_embeddings(
    chunks: list[dict],
    model_name: str = "all-MiniLM-L6-v2"
):
    """
    Generates embeddings for all document chunks.

    Args:
        chunks (list[dict]): List of text chunks.
        model_name (str): Pre-trained Sentence Transformer model.

    Returns:
        tuple:
            embeddings (numpy.ndarray): Vector representation of chunks.
            model (SentenceTransformer): Loaded embedding model.
    """

    # Load the embedding model
    print(f"Loading embedding model: {model_name}")

    model = SentenceTransformer(model_name)

    # Extract only the text from each chunk
    texts = [chunk["text"] for chunk in chunks]

    print(f"Generating embeddings for {len(texts)} text chunks...")

    # Generate embeddings
    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    print(f"Embedding dimension: {embeddings.shape[1]}")

    return embeddings, model


# ------------------------------------------------------------
# Function: Build FAISS Index
# Stores embeddings in a FAISS vector database for fast
# similarity search using cosine similarity.
# ------------------------------------------------------------

def build_faiss_index(embeddings: np.ndarray):
    """
    Builds a FAISS index from document embeddings.

    Args:
        embeddings (numpy.ndarray): Embedding vectors.

    Returns:
        faiss.IndexFlatIP: FAISS index containing all embeddings.
    """

    # Convert embeddings to float32 (required by FAISS)
    embeddings = np.asarray(embeddings, dtype=np.float32)

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    # Determine embedding dimension
    dimension = embeddings.shape[1]

    # Create a FAISS IndexFlatIP (Inner Product)
    # After normalization, inner product is equivalent to cosine similarity.
    index = faiss.IndexFlatIP(dimension)

    # Add all embeddings to the index
    index.add(embeddings)

    print(f"FAISS index created successfully with {index.ntotal} vectors.")

    return index

# **Step 5: Retrieve Relevant Chunks and Generate Answers**

After the document has been indexed, the Retrieval-Augmented Generation (RAG) system processes the user's question through three major steps.

### 1. Retrieve Relevant Document Chunks
- Convert the user's question into an embedding.
- Search the FAISS vector database using cosine similarity.
- Retrieve the top **k** most relevant text chunks along with their page numbers and similarity scores.

### 2. Build the Prompt
- Combine the retrieved document chunks into a single context.
- Append the user's question.
- Instruct the language model to answer **only** using the provided document context.

### 3. Generate the Final Answer
- Send the prompt to the Google Gemini Large Language Model (LLM).
- Generate a context-aware answer based only on the retrieved information.
- If the required information is not available in the document, the model responds accordingly instead of generating unsupported information.

This retrieval-first approach improves answer accuracy and reduces hallucinations compared to using a language model alone.

In [7]:
# ============================================================
# Step 5: Retrieve Relevant Chunks and Generate Answers
# ============================================================

# ------------------------------------------------------------
# Function: Retrieve Relevant Document Chunks
# Converts the user's query into an embedding and retrieves
# the most similar document chunks from the FAISS index.
# ------------------------------------------------------------

def retrieve_documents(query, index, chunks, model, k=3):
    """
    Retrieves the top-k most relevant document chunks.

    Args:
        query (str): User's question.
        index (faiss.Index): FAISS vector index.
        chunks (list): List of document chunks.
        model (SentenceTransformer): Embedding model.
        k (int): Number of chunks to retrieve.

    Returns:
        list: Retrieved document chunks with page numbers
              and similarity scores.
    """

    # Validate the user query
    if not query.strip():
        raise ValueError("Question cannot be empty.")

    # Generate embedding for the query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # Convert embedding to float32 for FAISS compatibility
    query_embedding = np.asarray(query_embedding, dtype=np.float32)

    # Normalize embedding for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search the FAISS index
    scores, indices = index.search(query_embedding, k)

    retrieved = []

    # Collect the retrieved chunks
    for score, idx in zip(scores[0], indices[0]):

        # Skip invalid indices
        if idx < 0:
            continue

        retrieved.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "score": float(score)
        })

    return retrieved


# ------------------------------------------------------------
# Function: Build Prompt
# Combines the retrieved document chunks and the user's
# question into a prompt for the Gemini model.
# ------------------------------------------------------------

def build_prompt(query, retrieved_results):
    """
    Creates a prompt using the retrieved document context.

    Args:
        query (str): User's question.
        retrieved_results (list): Retrieved document chunks.

    Returns:
        str: Prompt sent to the Gemini model.
    """

    # Build the document context
    context = ""

    for i, result in enumerate(retrieved_results, start=1):

        context += (
            f"\n--- Context {i} ---\n"
            f"Page: {result['page']}\n"
            f"Similarity Score: {result['score']:.4f}\n\n"
            f"{result['text']}\n"
        )

    # Create the prompt
    prompt = f"""
You are an intelligent AI assistant.

Answer the user's question ONLY using the retrieved document context.

If the required information is not available in the document, reply:

"The answer could not be found in the provided document."

Retrieved Document Context:
{context}

User Question:
{query}

Answer:
"""

    return prompt


# ------------------------------------------------------------
# Function: Generate Answer
# Sends the prompt to Google Gemini and returns the response.
# ------------------------------------------------------------

def generate_answer(prompt, api_key=None):
    """
    Generates an answer using Google Gemini.

    Args:
        prompt (str): Prompt containing retrieved context.
        api_key (str, optional): Google Gemini API key.

    Returns:
        str: Generated answer.
    """

    # Configure Gemini API if an API key is provided
    if api_key:
        genai.configure(api_key=api_key)

    # Load the Gemini model
    model = genai.GenerativeModel("gemini-2.5-flash")

    # Generate the response
    response = model.generate_content(prompt)

    # Return the generated text
    return response.text

# **Step 6: Execute the Complete RAG Pipeline**

This function combines all the individual components of the Retrieval-Augmented Generation (RAG) system into a single workflow.

The pipeline performs the following operations:

1. **Load the PDF document** and extract text.
2. **Clean the extracted text** to improve quality.
3. **Split the document** into smaller overlapping chunks.
4. **Generate semantic embeddings** for each text chunk.
5. **Build the FAISS vector database** for similarity search.
6. **Retrieve the most relevant document chunks** based on the user's question.
7. **Construct a prompt** using the retrieved context.
8. **Generate the final answer** using the Google Gemini Large Language Model.

The retrieved document chunks, along with their page numbers and similarity scores, are displayed before the generated answer to provide transparency and traceability.

In [8]:
# ============================================================
# Step 6: Execute the Complete RAG Pipeline
# ============================================================

# ------------------------------------------------------------
# Function: Run Complete RAG Pipeline
# Executes all stages of the Retrieval-Augmented Generation
# workflow from document loading to answer generation.
# ------------------------------------------------------------

def run_rag_pipeline(
    pdf_path: str,
    query: str,
    api_key: str,
    k: int = 3,
    chunk_size: int = 500,
    chunk_overlap: int = 50
):
    """
    Executes the complete Retrieval-Augmented Generation (RAG)
    pipeline.

    Args:
        pdf_path (str): Path to the PDF document.
        query (str): User's question.
        api_key (str): Google Gemini API key.
        k (int): Number of document chunks to retrieve.
        chunk_size (int): Maximum size of each text chunk.
        chunk_overlap (int): Number of overlapping characters.

    Returns:
        None
    """

    # --------------------------------------------------------
    # Validate user input
    # --------------------------------------------------------

    if not query.strip():
        print("Question cannot be empty.")
        return

    try:

        # ----------------------------------------------------
        # Step 1: Load PDF document
        # ----------------------------------------------------

        raw_pages = load_pdf(pdf_path)

        # ----------------------------------------------------
        # Step 2: Clean extracted text
        # ----------------------------------------------------

        cleaned_pages = clean_text(raw_pages)

        # ----------------------------------------------------
        # Step 3: Split text into chunks
        # ----------------------------------------------------

        chunks = split_text(
            cleaned_pages,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        # ----------------------------------------------------
        # Step 4: Generate embeddings
        # ----------------------------------------------------

        embeddings, model = create_embeddings(chunks)

        # ----------------------------------------------------
        # Step 5: Build FAISS vector index
        # ----------------------------------------------------

        index = build_faiss_index(embeddings)

        # ----------------------------------------------------
        # Step 6: Retrieve relevant document chunks
        # ----------------------------------------------------

        retrieved = retrieve_documents(
            query=query,
            index=index,
            chunks=chunks,
            model=model,
            k=k
        )

        # ----------------------------------------------------
        # Step 7: Build prompt
        # ----------------------------------------------------

        prompt = build_prompt(query, retrieved)

        # ----------------------------------------------------
        # Step 8: Generate answer using Gemini
        # ----------------------------------------------------

        answer = generate_answer(prompt, api_key)

        # ----------------------------------------------------
        # Display Retrieved Context
        # ----------------------------------------------------

        print("\n" + "=" * 80)
        print("Retrieved Context")
        print("=" * 80)

        for i, item in enumerate(retrieved, start=1):

            print(f"\nChunk {i}")
            print(f"Source Page      : {item['page']}")
            print(f"Similarity Score : {item['score']:.4f}")
            print("-" * 80)
            print(item["text"])
            print("-" * 80)

        # ----------------------------------------------------
        # Display Generated Answer
        # ----------------------------------------------------

        print("\n" + "=" * 80)
        print("Generated Answer")
        print("=" * 80)
        print(answer)

        print("=" * 80)

    # --------------------------------------------------------
    # Handle unexpected errors
    # --------------------------------------------------------

    except Exception as error:
        print(f"Pipeline Error: {error}")

# **Step 7: Document Processing and Index Creation**

Before users can ask questions, the PDF document must be processed and indexed.

This step performs the following operations:

1. **Accept the PDF file path** from the user.
2. **Validate** that the file exists.
3. **Load the PDF** and extract text from each page.
4. **Clean the extracted text** to improve readability.
5. **Split the document** into smaller overlapping text chunks.
6. **Generate semantic embeddings** for all text chunks.
7. **Build a FAISS vector index** for efficient similarity search.

This preprocessing stage is executed only once for each document. After the document has been indexed successfully, the system is ready to answer user queries.

In [ ]:
# ============================================================
# Step 7: Document Processing and Index Creation
# ============================================================

# ------------------------------------------------------------
# Accept the PDF file path from the user
# ------------------------------------------------------------

pdf_file_path = input("Enter the PDF file path: ").strip()

# ------------------------------------------------------------
# Initialize global variables
# These variables will be used during question answering.
# ------------------------------------------------------------

chunks = None
embeddings = None
embedding_model = None
faiss_index = None

# ------------------------------------------------------------
# Validate the PDF file path
# ------------------------------------------------------------

if not pdf_file_path:

    print("Error: PDF file path cannot be empty.")

elif not os.path.exists(pdf_file_path):

    print("Error: PDF file not found. Please check the file path.")

else:

    try:

        print("\nProcessing the PDF document...\n")

        # ----------------------------------------------------
        # Step 1: Load the PDF document
        # ----------------------------------------------------

        raw_pages = load_pdf(pdf_file_path)

        # ----------------------------------------------------
        # Step 2: Clean the extracted text
        # ----------------------------------------------------

        cleaned_pages = clean_text(raw_pages)

        # ----------------------------------------------------
        # Step 3: Split the document into text chunks
        # ----------------------------------------------------

        chunks = split_text(cleaned_pages)

        # ----------------------------------------------------
        # Step 4: Generate embeddings
        # ----------------------------------------------------

        embeddings, embedding_model = create_embeddings(chunks)

        # ----------------------------------------------------
        # Step 5: Build the FAISS vector index
        # ----------------------------------------------------

        faiss_index = build_faiss_index(embeddings)

        # ----------------------------------------------------
        # Display processing summary
        # ----------------------------------------------------

        print("\n" + "=" * 60)
        print("Document Processing Completed Successfully")
        print("=" * 60)
        print(f"PDF File        : {os.path.basename(pdf_file_path)}")
        print(f"Total Pages     : {len(cleaned_pages)}")
        print(f"Total Chunks    : {len(chunks)}")
        print(f"Vectors Indexed : {faiss_index.ntotal}")
        print("=" * 60)

    except Exception as error:

        print(f"\nDocument Processing Failed: {error}")

# **Step 8: Interactive Question Answering**

After the document has been processed and indexed, the Retrieval-Augmented Generation (RAG) system enters an interactive question-answering mode.

For each user question, the following steps are performed:

1. **Accept the user's question.**
2. **Convert the question into an embedding.**
3. **Retrieve the most relevant document chunks** from the FAISS vector database.
4. **Construct a prompt** using the retrieved context and the user's question.
5. **Generate an answer** using the Google Gemini Large Language Model.
6. **Display the retrieved document chunks**, including page numbers and similarity scores.
7. **Display the final generated answer.**

The interactive session continues until the user types **`exit`**.

In [ ]:
# ============================================================
# Step 8: Interactive Question Answering
# ============================================================

# Check whether the document has been processed successfully
if (
    pdf_file_path
    and os.path.exists(pdf_file_path)
    and faiss_index is not None
):

    print("=" * 80)
    print("Interactive RAG Chatbot")
    print("Type 'exit' to quit the chatbot.")
    print("=" * 80)

    # --------------------------------------------------------
    # Start the interactive question-answering loop
    # --------------------------------------------------------

    while True:

        # Accept a question from the user
        question = input("\nEnter your question: ").strip()

        # Exit the chatbot
        if question.lower() == "exit":
            print("\nThank you for using the RAG Chatbot. Goodbye!")
            break

        # Validate the question
        if not question:
            print("Question cannot be empty. Please try again.")
            continue

        try:

            # ------------------------------------------------
            # Step 1: Retrieve relevant document chunks
            # ------------------------------------------------

            retrieved = retrieve_documents(
                query=question,
                index=faiss_index,
                chunks=chunks,
                model=embedding_model,
                k=3
            )

            # ------------------------------------------------
            # Step 2: Build the prompt
            # ------------------------------------------------

            prompt = build_prompt(question, retrieved)

            # ------------------------------------------------
            # Step 3: Generate the answer
            # ------------------------------------------------

            answer = generate_answer(
                prompt,
                gemini_api_key
            )

            # ------------------------------------------------
            # Display retrieved document chunks
            # ------------------------------------------------

            print("\n" + "=" * 80)
            print("Retrieved Context")
            print("=" * 80)

            for i, chunk in enumerate(retrieved, start=1):

                print(f"\nChunk {i}")
                print(f"Source Page      : {chunk['page']}")
                print(f"Similarity Score : {chunk['score']:.4f}")
                print("-" * 80)
                print(chunk["text"])
                print("-" * 80)

            # ------------------------------------------------
            # Display generated answer
            # ------------------------------------------------

            print("\n" + "=" * 80)
            print("Generated Answer")
            print("=" * 80)
            print(answer)
            print("=" * 80)

        except Exception as error:

            print(f"\nError: {error}")

# ------------------------------------------------------------
# Display message if the document has not been processed
# ------------------------------------------------------------

else:

    print("Please process and index the PDF document before asking questions.")



# **Conclusion & Project Summary**

This project implements a **Retrieval-Augmented Generation (RAG)** system that answers questions from custom PDF documents using **Google Gemini**. Instead of relying only on pre-trained knowledge, the system retrieves relevant information from the document to generate accurate and context-aware responses.

# Getting Started

Before running the notebook, complete the following steps:

1. Install all the required Python packages.
2. Create a **Google Gemini API Key** from Google AI Studio.
3. Enter your API key when prompted during execution.
4. Keep the PDF document in the project folder or provide its full file path.
5. Execute the notebook cells sequentially from top to bottom for proper pipeline execution.
---
### Workflow

1. Load and extract text from the PDF.
2. Clean and split the text into chunks.
3. Generate embeddings using **Sentence Transformers**.
4. Store embeddings in a **FAISS** vector database.
5. Retrieve relevant chunks for a user query.
6. Generate the final answer using **Google Gemini**.

By combining semantic retrieval with language generation, the system provides reliable, document-grounded answers while reducing hallucinations.

---

## Key Features

- Supports custom PDF documents.
- Automatic text extraction and preprocessing.
- Semantic search using Sentence Transformers.
- Fast retrieval with FAISS.
- Context-aware answer generation using Google Gemini.
- Interactive question-answering interface.

---

## Applications

- Educational Question Answering
- Enterprise Knowledge Management
- Research Paper Analysis
- Legal Document Search
- Medical Documentation
- AI Chatbots
- Knowledge Base Systems

---

## Future Improvements

- Support multiple PDF documents.
- Implement hybrid (keyword + semantic) retrieval.
- Add conversation memory.
- Build a Streamlit/Gradio web interface.
- Use advanced embedding models and vector databases.

---

## Learning Outcomes

- Understanding the RAG architecture.
- PDF preprocessing and text chunking.
- Semantic embeddings with Sentence Transformers.
- FAISS vector indexing and retrieval.
- Google Gemini integration.
- End-to-end document question answering.

---

## Final Remarks

This project demonstrates a complete and modular **Document Question Answering System using RAG**. It provides a strong foundation for building intelligent document assistants, enterprise search systems, research assistants, and AI-powered knowledge management applications.